# Phase 4: Portfolio Construction & Risk Management  
## Notebook 04_11 — CVaR Convex Optimization

### Research question

How do we construct portfolios by directly minimising tail losses using Conditional Value at Risk, instead of relying only on variance or volatility?

This notebook follows:

```text
04_06_var_cvar_and_stress_testing
04_09_risk_parity_and_equal_risk_contribution
04_10_hierarchical_risk_parity
```

The previous notebooks measured CVaR and built risk-parity / HRP allocations. This notebook turns CVaR into an **optimisation objective**.

It covers:

1. VaR and CVaR definitions;
2. Rockafellar–Uryasev CVaR optimisation;
3. CVaR as a convex objective;
4. linear-programming formulation;
5. long-only CVaR minimisation;
6. target-return CVaR optimisation;
7. mean-CVaR efficient frontier;
8. comparison with equal weight, inverse vol, GMV, ERC, and HRP-style baselines;
9. scenario-based optimisation;
10. stress-scenario augmentation;
11. turnover-aware CVaR optimisation;
12. out-of-sample backtesting;
13. rolling walk-forward CVaR optimisation;
14. tail-risk and drawdown diagnostics;
15. governance flags;
16. saved outputs and manifest.

The key idea:

> Variance penalises upside and downside symmetrically. CVaR optimisation focuses directly on the average severity of bad-tail losses.

## 1. From VaR to CVaR

Define portfolio return:

$$
R_p = w^\top r
$$

Define loss:

$$
L = -R_p
$$

Value at Risk at confidence level $\alpha$:

$$
VaR_\alpha = q_\alpha(L)
$$

Conditional Value at Risk, also called Expected Shortfall:

$$
CVaR_\alpha = E[L \mid L \ge VaR_\alpha]
$$

VaR is the threshold.

CVaR is the average loss beyond the threshold.

For portfolio construction, CVaR is attractive because it focuses on the tail.

## 2. Rockafellar–Uryasev formulation

For scenarios $s=1,\dots,S$, portfolio loss:

$$
L_s(w)=-r_s^\top w
$$

Introduce threshold variable $\tau$, which plays the role of VaR.

Define:

$$
u_s \ge L_s(w)-\tau
$$

$$
u_s \ge 0
$$

Then empirical CVaR can be minimised through:

$$
\min_{w,\tau,u} \quad \tau + \frac{1}{(1-\alpha)S} \sum_{s=1}^{S}u_s
$$

subject to:

$$
u_s \ge -r_s^\top w-\tau
$$

$$
u_s \ge 0
$$

$$
\sum_i w_i=1
$$

$$
w_i \ge 0
$$

This is a linear programme when returns are scenario samples.

That is the magic: **CVaR optimisation is convex and can be solved reliably.**

## 3. Why CVaR optimisation matters

Mean-variance optimisation solves:

$$
\min_w w^\top\Sigma w
$$

It controls average quadratic fluctuations.

CVaR optimisation solves:

$$
\min_w CVaR_\alpha(-w^\top r)
$$

It controls the average bad-tail loss.

These are different.

A portfolio with low volatility can still have bad crash exposure.

A portfolio with slightly higher volatility may have better tail behaviour.

This notebook compares those trade-offs.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.optimize import linprog, minimize
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

try:
    from sklearn.covariance import LedoitWolf
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

SCIPY_AVAILABLE, SKLEARN_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class CVaROptimizationConfig:
    n_days: int = 2_500
    train_fraction: float = 0.65
    annualisation: int = 252
    alpha: float = 0.95
    max_weight: float = 0.35
    min_weight: float = 0.00
    target_return_grid_points: int = 18
    rolling_window: int = 504
    rebalance_step: int = 21
    transaction_cost_bps: float = 3.0
    stress_scenario_multiplier: int = 8
    turnover_penalty: float = 0.002
    initial_capital: float = 1_000_000.0
    seed: int = 42


config = CVaROptimizationConfig()
config

## 5. Synthetic multi-asset universe with fat tails

CVaR optimisation needs realistic tail scenarios.

We simulate returns with:

1. multi-asset factor structure;
2. volatility regimes;
3. fat-tailed shocks;
4. equity crashes;
5. inflation shocks;
6. commodity crashes;
7. crypto crashes;
8. trend-following crisis alpha.

This gives the optimiser non-Gaussian downside to react to.

In [ ]:
def simulate_cvar_universe(config: CVaROptimizationConfig):
    rng = np.random.default_rng(config.seed)
    dates = pd.bdate_range("2014-01-01", periods=config.n_days)

    assets = [
        "US_EQ", "EU_EQ", "EM_EQ",
        "US_BOND", "EU_BOND",
        "GOLD", "OIL", "COPPER",
        "FX_CARRY", "TREND",
        "REIT", "BTC_PROXY"
    ]

    asset_class = {
        "US_EQ": "equity",
        "EU_EQ": "equity",
        "EM_EQ": "equity",
        "US_BOND": "bond",
        "EU_BOND": "bond",
        "GOLD": "commodity",
        "OIL": "commodity",
        "COPPER": "commodity",
        "FX_CARRY": "fx",
        "TREND": "alternative",
        "REIT": "real_asset",
        "BTC_PROXY": "crypto",
    }

    factor_names = ["market", "rates", "commodity", "carry", "trend", "crypto"]
    factor_vol_ann = np.array([0.14, 0.055, 0.13, 0.08, 0.09, 0.38])
    factor_vol_daily = factor_vol_ann / np.sqrt(config.annualisation)

    factor_corr = np.array([
        [ 1.00, -0.25,  0.25,  0.15, -0.15,  0.35],
        [-0.25,  1.00,  0.10, -0.05,  0.15, -0.20],
        [ 0.25,  0.10,  1.00,  0.10,  0.05,  0.20],
        [ 0.15, -0.05,  0.10,  1.00,  0.00,  0.15],
        [-0.15,  0.15,  0.05,  0.00,  1.00,  0.00],
        [ 0.35, -0.20,  0.20,  0.15,  0.00,  1.00],
    ])

    factor_cov = np.outer(factor_vol_daily, factor_vol_daily) * factor_corr

    loadings = pd.DataFrame(0.0, index=assets, columns=factor_names)
    loadings.loc[["US_EQ", "EU_EQ", "EM_EQ", "REIT"], "market"] = [1.00, 0.95, 1.20, 0.75]
    loadings.loc[["US_BOND", "EU_BOND"], "rates"] = [1.00, 0.90]
    loadings.loc[["GOLD", "OIL", "COPPER"], "commodity"] = [0.45, 1.10, 0.95]
    loadings.loc["FX_CARRY", "carry"] = 1.00
    loadings.loc["TREND", "trend"] = 1.00
    loadings.loc["TREND", "market"] = -0.30
    loadings.loc["BTC_PROXY", "crypto"] = 1.00
    loadings.loc["BTC_PROXY", "market"] = 0.40
    loadings.loc["GOLD", "rates"] = 0.25
    loadings.loc["REIT", "rates"] = -0.35

    idio_vol_ann = pd.Series({
        "US_EQ": 0.07,
        "EU_EQ": 0.08,
        "EM_EQ": 0.13,
        "US_BOND": 0.025,
        "EU_BOND": 0.030,
        "GOLD": 0.12,
        "OIL": 0.22,
        "COPPER": 0.18,
        "FX_CARRY": 0.07,
        "TREND": 0.08,
        "REIT": 0.11,
        "BTC_PROXY": 0.52,
    })

    annual_mean = pd.Series({
        "US_EQ": 0.07,
        "EU_EQ": 0.06,
        "EM_EQ": 0.08,
        "US_BOND": 0.025,
        "EU_BOND": 0.020,
        "GOLD": 0.035,
        "OIL": 0.045,
        "COPPER": 0.040,
        "FX_CARRY": 0.030,
        "TREND": 0.050,
        "REIT": 0.060,
        "BTC_PROXY": 0.120,
    })

    regime = np.zeros(config.n_days, dtype=int)
    transition = np.array([[0.985, 0.015], [0.055, 0.945]])

    factor_returns = np.zeros((config.n_days, len(factor_names)))
    asset_returns = np.zeros((config.n_days, len(assets)))
    stress_type = np.array(["normal"] * config.n_days, dtype=object)

    for t in range(config.n_days):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_multiplier = 1.0 if regime[t] == 0 else 2.4
        cov_t = factor_cov * vol_multiplier**2

        z = rng.standard_t(df=5, size=len(factor_names)) * np.sqrt((5 - 2) / 5)
        L = np.linalg.cholesky(cov_t + 1e-12 * np.eye(len(factor_names)))
        f = L @ z

        u = rng.uniform()
        if u < 0.010:
            stress_type[t] = "equity_crash"
            f[0] -= rng.uniform(0.040, 0.120)
            f[1] += rng.uniform(0.004, 0.030)
            f[5] -= rng.uniform(0.080, 0.250)
            f[4] += rng.uniform(0.005, 0.045)
        elif u < 0.017:
            stress_type[t] = "inflation_shock"
            f[1] -= rng.uniform(0.010, 0.045)
            f[2] += rng.uniform(0.040, 0.120)
            f[0] -= rng.uniform(0.010, 0.055)
        elif u < 0.024:
            stress_type[t] = "commodity_crash"
            f[2] -= rng.uniform(0.050, 0.150)
            f[0] -= rng.uniform(0.005, 0.035)
            f[4] += rng.uniform(0.005, 0.035)
        elif u < 0.031:
            stress_type[t] = "crypto_crash"
            f[5] -= rng.uniform(0.150, 0.350)
            f[0] -= rng.uniform(0.005, 0.030)

        eps = rng.standard_t(df=4, size=len(assets)) * np.sqrt((4 - 2) / 4)
        eps = eps * (idio_vol_ann.values / np.sqrt(config.annualisation))

        mu_daily = annual_mean.values / config.annualisation
        asset_returns[t] = mu_daily + loadings.to_numpy() @ f + eps
        factor_returns[t] = f

    returns_df = pd.DataFrame(asset_returns, columns=assets)
    returns_df.insert(0, "date", dates)
    returns_df["regime"] = regime
    returns_df["stress_type"] = stress_type

    factor_df = pd.DataFrame(factor_returns, columns=factor_names)
    factor_df.insert(0, "date", dates)
    factor_df["regime"] = regime
    factor_df["stress_type"] = stress_type

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "idio_vol_ann": [idio_vol_ann[a] for a in assets],
        "annual_mean_assumption": [annual_mean[a] for a in assets],
    })

    return returns_df, factor_df, metadata


returns_df, factor_returns, asset_metadata = simulate_cvar_universe(config)
asset_cols = asset_metadata["asset"].tolist()
returns = returns_df.set_index("date")[asset_cols]

returns_df.head(), asset_metadata

In [ ]:
plt.figure(figsize=(12, 6))
for asset in asset_cols:
    plt.plot(returns.index, (1 + returns[asset]).cumprod(), label=asset, alpha=0.75)
plt.title("Synthetic Asset Growth of 1")
plt.xlabel("Date")
plt.ylabel("Growth")
plt.legend(ncol=3)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(returns_df["date"], returns_df["regime"], drawstyle="steps-post")
plt.title("Volatility Regime")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 6. Return diagnostics

We inspect return moments and tail behaviour.

CVaR optimisation is most useful when returns are skewed, fat-tailed, or crisis-sensitive.

In [ ]:
return_summary = returns.agg(["mean", "std", "min", "max"]).T.rename(columns={
    "mean": "daily_mean",
    "std": "daily_vol",
    "min": "worst_day",
    "max": "best_day",
})

return_summary["annualised_mean"] = return_summary["daily_mean"] * config.annualisation
return_summary["annualised_vol"] = return_summary["daily_vol"] * np.sqrt(config.annualisation)
return_summary["skew"] = returns.skew()
return_summary["excess_kurtosis"] = returns.kurtosis()
return_summary["asset_class"] = return_summary.index.map(asset_metadata.set_index("asset")["asset_class"])

return_summary.sort_values("worst_day")

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(returns["US_EQ"], bins=100, alpha=0.5, density=True, label="US_EQ")
plt.hist(returns["TREND"], bins=100, alpha=0.5, density=True, label="TREND")
plt.hist(returns["BTC_PROXY"], bins=100, alpha=0.5, density=True, label="BTC_PROXY")
plt.axvline(0, linestyle="--")
plt.title("Return Distributions")
plt.xlabel("Daily return")
plt.ylabel("Density")
plt.legend()
plt.show()

## 7. Train/test split

CVaR optimisation can overfit historical tail scenarios.

We estimate allocations on train data and evaluate on unseen test data.

In [ ]:
split_idx = int(len(returns) * config.train_fraction)

train_returns = returns.iloc[:split_idx]
test_returns = returns.iloc[split_idx:]

split_metadata = {
    "n_total_days": int(len(returns)),
    "n_train_days": int(len(train_returns)),
    "n_test_days": int(len(test_returns)),
    "train_start": str(train_returns.index[0]),
    "train_end": str(train_returns.index[-1]),
    "test_start": str(test_returns.index[0]),
    "test_end": str(test_returns.index[-1]),
}

pd.Series(split_metadata)

## 8. CVaR and performance utilities

We define:

$$
L_t = -R_{p,t}
$$

and estimate VaR/CVaR empirically.

In [ ]:
def portfolio_returns_from_weights(returns_df, weights):
    return returns_df @ pd.Series(weights, index=returns_df.columns)


def historical_var_cvar(losses, alpha):
    losses = np.asarray(losses, dtype=float)
    var = np.quantile(losses, alpha)
    tail = losses[losses >= var]
    cvar = tail.mean() if len(tail) else np.nan
    return float(var), float(cvar)


def portfolio_tail_metrics(returns_df, weights, alpha):
    r = portfolio_returns_from_weights(returns_df, weights)
    losses = -r
    var, cvar = historical_var_cvar(losses, alpha)

    return {
        "mean_daily": float(r.mean()),
        "vol_daily": float(r.std(ddof=1)),
        "VaR": var,
        "CVaR": cvar,
        "worst_day": float(r.min()),
        "skew": float(r.skew()),
        "excess_kurtosis": float(r.kurtosis())
    }


def max_drawdown_from_returns(r):
    nav = (1 + r).cumprod()
    dd = nav / nav.cummax() - 1.0
    return dd, float(dd.min())

## 9. CVaR linear programming solver

Variables:

$$
x = [w_1,\dots,w_N,\tau,u_1,\dots,u_S]
$$

Objective:

$$
\min \quad \tau + \frac{1}{(1-\alpha)S} \sum_s u_s
$$

Constraints:

$$
-r_s^\top w-\tau-u_s \le 0
$$

$$
\sum_i w_i = 1
$$

$$
w_i \in [w_{min},w_{max}]
$$

$$
u_s \ge 0
$$

Optional target return:

$$
w^\top \hat \mu \ge \mu^*
$$

In [ ]:
def solve_cvar_lp(
    returns_scenarios,
    alpha=0.95,
    min_weight=0.0,
    max_weight=0.35,
    target_return_daily=None,
    previous_weights=None,
    turnover_penalty=0.0,
):
    R = np.asarray(returns_scenarios, dtype=float)
    n_scenarios, n_assets = R.shape

    # Variables: weights n_assets, tau 1, u n_scenarios.
    n_vars = n_assets + 1 + n_scenarios
    tau_idx = n_assets
    u_start = n_assets + 1

    c = np.zeros(n_vars)
    c[tau_idx] = 1.0
    c[u_start:] = 1.0 / ((1 - alpha) * n_scenarios)

    # Optional simple turnover penalty linearised with auxiliary variables would require extra variables.
    # For this first LP, turnover is handled in a separate approximate routine.
    if turnover_penalty != 0.0:
        raise NotImplementedError("Use solve_cvar_lp_with_turnover for turnover-aware optimisation.")

    # Inequality: -R_s w - tau - u_s <= 0.
    A_ub = []
    b_ub = []

    for s in range(n_scenarios):
        row = np.zeros(n_vars)
        row[:n_assets] = -R[s]
        row[tau_idx] = -1.0
        row[u_start + s] = -1.0
        A_ub.append(row)
        b_ub.append(0.0)

    # Optional target return: -mu'w <= -target.
    mu = R.mean(axis=0)
    if target_return_daily is not None:
        row = np.zeros(n_vars)
        row[:n_assets] = -mu
        A_ub.append(row)
        b_ub.append(-target_return_daily)

    A_ub = np.asarray(A_ub)
    b_ub = np.asarray(b_ub)

    # Equality sum weights = 1.
    A_eq = np.zeros((1, n_vars))
    A_eq[0, :n_assets] = 1.0
    b_eq = np.array([1.0])

    bounds = []
    for _ in range(n_assets):
        bounds.append((min_weight, max_weight))
    bounds.append((None, None))  # tau
    for _ in range(n_scenarios):
        bounds.append((0.0, None))  # u

    if SCIPY_AVAILABLE:
        res = linprog(
            c,
            A_ub=A_ub,
            b_ub=b_ub,
            A_eq=A_eq,
            b_eq=b_eq,
            bounds=bounds,
            method="highs",
        )

        if res.success:
            w = res.x[:n_assets]
            tau = res.x[tau_idx]
            u = res.x[u_start:]
            return {
                "success": True,
                "method": "scipy_linprog_highs",
                "weights": w,
                "tau": tau,
                "u": u,
                "objective": float(res.fun),
                "message": res.message,
            }

    # Fallback random search.
    rng = np.random.default_rng(config.seed)
    best_w = np.ones(n_assets) / n_assets
    best_obj = np.inf
    best_tau = np.nan

    for _ in range(50_000):
        w = rng.dirichlet(np.ones(n_assets))
        w = np.clip(w, min_weight, max_weight)
        w = w / w.sum()

        if target_return_daily is not None and w @ mu < target_return_daily:
            continue

        losses = -R @ w
        var, cvar = historical_var_cvar(losses, alpha)

        if cvar < best_obj:
            best_obj = cvar
            best_w = w
            best_tau = var

    return {
        "success": False,
        "method": "fallback_random_search",
        "weights": best_w,
        "tau": best_tau,
        "u": np.maximum(-R @ best_w - best_tau, 0.0),
        "objective": float(best_obj),
        "message": "scipy unavailable or LP failed; used random search fallback",
    }

## 10. Baseline allocations

We compare CVaR optimisation against:

1. equal weight;
2. inverse volatility;
3. global minimum variance;
4. HRP-style recursive bisection fallback;
5. minimum CVaR portfolio.

The HRP implementation is simplified but captures recursive clustering logic when SciPy clustering is available.

In [ ]:
def estimate_covariance(returns_window):
    X = returns_window.dropna().to_numpy()

    if SKLEARN_AVAILABLE:
        estimator = LedoitWolf().fit(X)
        cov = estimator.covariance_
        method = "sklearn_ledoit_wolf"
        shrinkage = float(estimator.shrinkage_)
    else:
        sample = np.cov(X, rowvar=False, ddof=1)
        target = np.diag(np.diag(sample))
        shrinkage = 0.30
        cov = shrinkage * target + (1 - shrinkage) * sample
        method = "fallback_diagonal_shrinkage"

    return pd.DataFrame(cov, index=returns_window.columns, columns=returns_window.columns), method, shrinkage


def inverse_vol_weights(cov_df):
    vols = np.sqrt(np.diag(cov_df.to_numpy()))
    inv = 1.0 / np.maximum(vols, 1e-12)
    return pd.Series(inv / inv.sum(), index=cov_df.index)


def portfolio_volatility(weights, cov):
    w = np.asarray(weights, dtype=float)
    cov_arr = np.asarray(cov, dtype=float)
    return float(np.sqrt(max(w.T @ cov_arr @ w, 0.0)))


def gmv_weights(cov_df, max_weight=0.35):
    n = cov_df.shape[0]
    cov = cov_df.to_numpy()
    x0 = np.ones(n) / n

    def obj(w):
        return float(w.T @ cov @ w)

    if SCIPY_AVAILABLE:
        cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
        bounds = [(0.0, max_weight)] * n
        res = minimize(obj, x0=x0, method="SLSQP", bounds=bounds, constraints=cons, options={"maxiter": 1000, "ftol": 1e-12})
        if res.success:
            return pd.Series(res.x / res.x.sum(), index=cov_df.index)

    # fallback inverse covariance then clip
    inv_cov = np.linalg.pinv(cov)
    raw = inv_cov @ np.ones(n)
    raw = np.clip(raw, 0.0, max_weight)
    if raw.sum() <= 0:
        raw = np.ones(n) / n
    else:
        raw = raw / raw.sum()
    return pd.Series(raw, index=cov_df.index)


def simple_hrp_weights(cov_df, corr_df):
    # Dependency-light recursive allocation using greedy ordered assets.
    dist = np.sqrt(np.maximum(0.0, (1.0 - corr_df) / 2.0))
    remaining = list(corr_df.index)
    ordered = [remaining.pop(0)]
    while remaining:
        last = ordered[-1]
        next_asset = min(remaining, key=lambda x: dist.loc[last, x])
        ordered.append(next_asset)
        remaining.remove(next_asset)

    def inv_var_cluster_var(cluster):
        sub_cov = cov_df.loc[cluster, cluster].to_numpy()
        iv = 1.0 / np.maximum(np.diag(sub_cov), 1e-12)
        w = iv / iv.sum()
        return float(w.T @ sub_cov @ w)

    weights = pd.Series(1.0, index=ordered)
    clusters = [ordered]

    while clusters:
        new_clusters = []
        for cluster in clusters:
            if len(cluster) <= 1:
                continue
            split = len(cluster) // 2
            left = cluster[:split]
            right = cluster[split:]

            var_left = inv_var_cluster_var(left)
            var_right = inv_var_cluster_var(right)

            alpha = 1.0 - var_left / (var_left + var_right) if (var_left + var_right) > 0 else 0.5

            weights[left] *= alpha
            weights[right] *= 1.0 - alpha

            new_clusters.append(left)
            new_clusters.append(right)
        clusters = new_clusters

    weights = weights / weights.sum()
    return weights.reindex(cov_df.index).fillna(0.0)


cov_train, cov_method, cov_shrinkage = estimate_covariance(train_returns)
corr_train = train_returns.corr()

w_equal = pd.Series(1.0 / len(asset_cols), index=asset_cols)
w_inverse_vol = inverse_vol_weights(cov_train)
w_gmv = gmv_weights(cov_train, max_weight=config.max_weight)
w_hrp = simple_hrp_weights(cov_train, corr_train)

cvar_solution = solve_cvar_lp(
    train_returns,
    alpha=config.alpha,
    min_weight=config.min_weight,
    max_weight=config.max_weight
)
w_cvar = pd.Series(cvar_solution["weights"], index=asset_cols)

weights_static = pd.DataFrame({
    "equal_weight": w_equal,
    "inverse_vol": w_inverse_vol,
    "global_min_variance": w_gmv,
    "simple_hrp": w_hrp,
    "min_cvar": w_cvar,
})

weights_static

In [ ]:
plt.figure(figsize=(12, 7))
bottom = np.zeros(weights_static.shape[1])
x = np.arange(weights_static.shape[1])

for asset in weights_static.index:
    plt.bar(x, weights_static.loc[asset], bottom=bottom, label=asset)
    bottom += weights_static.loc[asset].values

plt.xticks(x, weights_static.columns, rotation=45, ha="right")
plt.title("Static Portfolio Weights")
plt.ylabel("Weight")
plt.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

## 11. In-sample tail-risk comparison

We compare training-sample tail metrics.

This is not the final judgment, but it confirms the optimisation objective is being targeted.

In [ ]:
def allocation_summary(returns_window, weights_df, alpha):
    rows = []

    cov = returns_window.cov().to_numpy()

    for name in weights_df.columns:
        w = weights_df[name]
        r = returns_window @ w
        losses = -r
        var, cvar = historical_var_cvar(losses, alpha)
        nav = (1 + r).cumprod()
        dd, mdd = max_drawdown_from_returns(r)

        rows.append({
            "portfolio": name,
            "mean_ann": float(r.mean() * config.annualisation),
            "vol_ann": float(r.std(ddof=1) * np.sqrt(config.annualisation)),
            "VaR": var,
            "CVaR": cvar,
            "worst_day": float(r.min()),
            "max_drawdown": mdd,
            "sharpe_proxy": float(r.mean() / r.std(ddof=1) * np.sqrt(config.annualisation)) if r.std(ddof=1) > 0 else np.nan,
            "effective_n": float(1.0 / np.sum(w.to_numpy() ** 2)),
            "max_weight": float(w.max()),
        })

    return pd.DataFrame(rows).sort_values("CVaR")


train_allocation_summary = allocation_summary(train_returns, weights_static, config.alpha)
train_allocation_summary

## 12. Out-of-sample test

The real question:

> Does a training-sample CVaR minimiser reduce tail risk in unseen data?

We apply static weights to the test sample.

In [ ]:
test_allocation_summary = allocation_summary(test_returns, weights_static, config.alpha)
test_allocation_summary

In [ ]:
plt.figure(figsize=(12, 6))
for name in weights_static.columns:
    r = test_returns @ weights_static[name]
    plt.plot(r.index, (1 + r).cumprod(), label=name)
plt.title("Static Out-of-Sample Growth")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plot_df = test_allocation_summary.sort_values("CVaR")
plt.barh(plot_df["portfolio"], plot_df["CVaR"])
plt.title(f"Out-of-Sample {int(config.alpha*100)}% CVaR")
plt.xlabel("CVaR loss")
plt.ylabel("Portfolio")
plt.show()

## 13. Mean-CVaR efficient frontier

A target-return CVaR frontier solves:

$$
\min_w CVaR_\alpha(-w^\top r)
$$

subject to:

$$
w^\top\hat\mu \ge \mu^*
$$

$$
\sum_i w_i=1
$$

$$
0\le w_i\le w_{max}
$$

This shows the trade-off between expected return and tail loss.

In [ ]:
mu_train = train_returns.mean()

target_min = np.percentile(mu_train, 20)
target_max = np.percentile(mu_train, 90)
target_grid = np.linspace(target_min, target_max, config.target_return_grid_points)

frontier_rows = []
frontier_weights = []

for i, target in enumerate(target_grid):
    sol = solve_cvar_lp(
        train_returns,
        alpha=config.alpha,
        min_weight=config.min_weight,
        max_weight=config.max_weight,
        target_return_daily=target,
    )

    w = pd.Series(sol["weights"], index=asset_cols)
    metrics_train = portfolio_tail_metrics(train_returns, w, config.alpha)
    metrics_test = portfolio_tail_metrics(test_returns, w, config.alpha)

    frontier_rows.append({
        "frontier_id": i,
        "target_return_daily": target,
        "target_return_ann": target * config.annualisation,
        "success": sol["success"],
        "method": sol["method"],
        "objective": sol["objective"],
        "train_mean_ann": metrics_train["mean_daily"] * config.annualisation,
        "train_vol_ann": metrics_train["vol_daily"] * np.sqrt(config.annualisation),
        "train_VaR": metrics_train["VaR"],
        "train_CVaR": metrics_train["CVaR"],
        "test_mean_ann": metrics_test["mean_daily"] * config.annualisation,
        "test_vol_ann": metrics_test["vol_daily"] * np.sqrt(config.annualisation),
        "test_VaR": metrics_test["VaR"],
        "test_CVaR": metrics_test["CVaR"],
    })

    weight_record = w.to_dict()
    weight_record["frontier_id"] = i
    frontier_weights.append(weight_record)

mean_cvar_frontier = pd.DataFrame(frontier_rows)
mean_cvar_frontier_weights = pd.DataFrame(frontier_weights)

mean_cvar_frontier.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(mean_cvar_frontier["train_CVaR"], mean_cvar_frontier["train_mean_ann"], marker="o", label="train")
plt.plot(mean_cvar_frontier["test_CVaR"], mean_cvar_frontier["test_mean_ann"], marker="x", label="test")
plt.title("Mean-CVaR Frontier")
plt.xlabel("CVaR loss")
plt.ylabel("Annualised mean return")
plt.legend()
plt.show()

## 14. Stress-scenario augmentation

A CVaR optimiser can underweight historical crises if they are rare.

One remedy is to augment the scenario set with repeated stress scenarios.

We create synthetic stress rows and append them multiple times to the training set.

This makes stress outcomes more important in the optimisation.

In [ ]:
def build_stress_scenarios(asset_cols):
    scenarios = {
        "equity_crash": {
            "US_EQ": -0.085, "EU_EQ": -0.090, "EM_EQ": -0.120,
            "US_BOND": 0.015, "EU_BOND": 0.012,
            "GOLD": 0.015, "OIL": -0.040, "COPPER": -0.055,
            "FX_CARRY": -0.025, "TREND": 0.030,
            "REIT": -0.080, "BTC_PROXY": -0.220,
        },
        "inflation_shock": {
            "US_EQ": -0.040, "EU_EQ": -0.045, "EM_EQ": -0.060,
            "US_BOND": -0.035, "EU_BOND": -0.030,
            "GOLD": 0.040, "OIL": 0.095, "COPPER": 0.075,
            "FX_CARRY": 0.005, "TREND": 0.020,
            "REIT": -0.075, "BTC_PROXY": -0.080,
        },
        "commodity_crash": {
            "US_EQ": -0.020, "EU_EQ": -0.025, "EM_EQ": -0.040,
            "US_BOND": 0.006, "EU_BOND": 0.005,
            "GOLD": -0.020, "OIL": -0.130, "COPPER": -0.110,
            "FX_CARRY": -0.020, "TREND": 0.025,
            "REIT": -0.020, "BTC_PROXY": -0.070,
        },
        "crypto_crash": {
            "US_EQ": -0.015, "EU_EQ": -0.015, "EM_EQ": -0.020,
            "US_BOND": 0.002, "EU_BOND": 0.002,
            "GOLD": 0.004, "OIL": -0.005, "COPPER": -0.005,
            "FX_CARRY": -0.005, "TREND": 0.006,
            "REIT": -0.010, "BTC_PROXY": -0.350,
        },
        "everything_sells_off": {
            "US_EQ": -0.080, "EU_EQ": -0.085, "EM_EQ": -0.110,
            "US_BOND": -0.025, "EU_BOND": -0.025,
            "GOLD": -0.030, "OIL": -0.080, "COPPER": -0.090,
            "FX_CARRY": -0.040, "TREND": 0.010,
            "REIT": -0.090, "BTC_PROXY": -0.250,
        },
    }

    return pd.DataFrame(scenarios).T.reindex(columns=asset_cols)


stress_scenarios = build_stress_scenarios(asset_cols)

stress_scenarios

In [ ]:
def augment_with_stress_scenarios(train_returns, stress_scenarios, multiplier):
    stress_rows = []
    for _ in range(multiplier):
        stress_rows.append(stress_scenarios.copy())

    stress_block = pd.concat(stress_rows, ignore_index=True)
    stress_block.index = [f"stress_{i}" for i in range(len(stress_block))]

    augmented = pd.concat([train_returns.copy(), stress_block], axis=0)
    return augmented


augmented_train = augment_with_stress_scenarios(
    train_returns,
    stress_scenarios,
    config.stress_scenario_multiplier
)

stress_cvar_solution = solve_cvar_lp(
    augmented_train,
    alpha=config.alpha,
    min_weight=config.min_weight,
    max_weight=config.max_weight
)

w_stress_cvar = pd.Series(stress_cvar_solution["weights"], index=asset_cols)

weights_static["stress_augmented_cvar"] = w_stress_cvar

weights_static[["min_cvar", "stress_augmented_cvar"]]

## 15. Scenario stress comparison

We evaluate portfolio losses under the explicit stress scenarios.

A stress-augmented optimiser should usually reduce losses under the added scenarios, but may sacrifice return or diversify less naturally.

In [ ]:
def scenario_losses(weights_df, stress_scenarios):
    rows = []

    for scenario, shock in stress_scenarios.iterrows():
        for portfolio in weights_df.columns:
            w = weights_df[portfolio]
            ret = float(w @ shock)
            rows.append({
                "scenario": scenario,
                "portfolio": portfolio,
                "scenario_return": ret,
                "scenario_loss": -ret,
            })

    return pd.DataFrame(rows)


stress_loss_table = scenario_losses(weights_static, stress_scenarios)
stress_loss_table.sort_values("scenario_loss", ascending=False).head(20)

In [ ]:
plt.figure(figsize=(12, 6))
for portfolio in ["equal_weight", "min_cvar", "stress_augmented_cvar", "simple_hrp"]:
    sub = stress_loss_table[stress_loss_table["portfolio"] == portfolio]
    plt.plot(sub["scenario"], sub["scenario_loss"], marker="o", label=portfolio)
plt.xticks(rotation=45, ha="right")
plt.title("Stress Scenario Losses")
plt.ylabel("Scenario loss")
plt.legend()
plt.tight_layout()
plt.show()

## 16. Turnover-aware CVaR approximation

A fully linear turnover-aware CVaR optimiser introduces auxiliary variables for absolute turnover.

We implement:

$$
\min CVaR + \lambda \sum_i |w_i-w_i^{prev}|
$$

using extra variables $v_i$:

$$
v_i \ge w_i-w_i^{prev}
$$

$$
v_i \ge -(w_i-w_i^{prev})
$$

$$
v_i \ge 0
$$

This is still a linear programme.

In [ ]:
def solve_cvar_lp_with_turnover(
    returns_scenarios,
    previous_weights,
    alpha=0.95,
    min_weight=0.0,
    max_weight=0.35,
    turnover_penalty=0.002,
):
    R = np.asarray(returns_scenarios, dtype=float)
    prev = np.asarray(previous_weights, dtype=float)

    n_scenarios, n_assets = R.shape

    # Variables: w n, tau 1, u S, v n.
    n_vars = n_assets + 1 + n_scenarios + n_assets
    tau_idx = n_assets
    u_start = n_assets + 1
    v_start = n_assets + 1 + n_scenarios

    c = np.zeros(n_vars)
    c[tau_idx] = 1.0
    c[u_start:u_start+n_scenarios] = 1.0 / ((1 - alpha) * n_scenarios)
    c[v_start:] = turnover_penalty

    A_ub = []
    b_ub = []

    # CVaR constraints: -R_s w - tau - u_s <= 0.
    for s in range(n_scenarios):
        row = np.zeros(n_vars)
        row[:n_assets] = -R[s]
        row[tau_idx] = -1.0
        row[u_start + s] = -1.0
        A_ub.append(row)
        b_ub.append(0.0)

    # Turnover variables:
    # w_i - prev_i - v_i <= 0
    # -w_i + prev_i - v_i <= 0
    for i in range(n_assets):
        row = np.zeros(n_vars)
        row[i] = 1.0
        row[v_start + i] = -1.0
        A_ub.append(row)
        b_ub.append(prev[i])

        row = np.zeros(n_vars)
        row[i] = -1.0
        row[v_start + i] = -1.0
        A_ub.append(row)
        b_ub.append(-prev[i])

    A_ub = np.asarray(A_ub)
    b_ub = np.asarray(b_ub)

    A_eq = np.zeros((1, n_vars))
    A_eq[0, :n_assets] = 1.0
    b_eq = np.array([1.0])

    bounds = []
    for _ in range(n_assets):
        bounds.append((min_weight, max_weight))
    bounds.append((None, None))  # tau
    for _ in range(n_scenarios):
        bounds.append((0.0, None))  # u
    for _ in range(n_assets):
        bounds.append((0.0, None))  # v

    if SCIPY_AVAILABLE:
        res = linprog(
            c,
            A_ub=A_ub,
            b_ub=b_ub,
            A_eq=A_eq,
            b_eq=b_eq,
            bounds=bounds,
            method="highs",
        )

        if res.success:
            return {
                "success": True,
                "method": "scipy_linprog_highs_turnover",
                "weights": res.x[:n_assets],
                "tau": res.x[tau_idx],
                "objective": float(res.fun),
                "turnover": float(np.sum(res.x[v_start:])),
                "message": res.message,
            }

    # fallback: blend min-CVaR solution with previous weights
    sol = solve_cvar_lp(
        returns_scenarios,
        alpha=alpha,
        min_weight=min_weight,
        max_weight=max_weight
    )
    raw = pd.Series(sol["weights"])
    blended = 0.5 * raw.to_numpy() + 0.5 * prev
    blended = np.clip(blended, min_weight, max_weight)
    blended = blended / blended.sum()

    return {
        "success": False,
        "method": "fallback_blend_previous_and_cvar",
        "weights": blended,
        "tau": np.nan,
        "objective": np.nan,
        "turnover": float(np.sum(np.abs(blended - prev))),
        "message": "linprog unavailable or failed",
    }


previous_w = weights_static["equal_weight"].to_numpy()

turnover_cvar_solution = solve_cvar_lp_with_turnover(
    train_returns,
    previous_weights=previous_w,
    alpha=config.alpha,
    min_weight=config.min_weight,
    max_weight=config.max_weight,
    turnover_penalty=config.turnover_penalty
)

w_turnover_cvar = pd.Series(turnover_cvar_solution["weights"], index=asset_cols)
weights_static["turnover_aware_cvar"] = w_turnover_cvar

weights_static[["equal_weight", "min_cvar", "turnover_aware_cvar"]]

## 17. Updated static comparison

Include stress-augmented and turnover-aware CVaR portfolios.

In [ ]:
train_allocation_summary = allocation_summary(train_returns, weights_static, config.alpha)
test_allocation_summary = allocation_summary(test_returns, weights_static, config.alpha)

test_allocation_summary

In [ ]:
plt.figure(figsize=(12, 6))
for name in weights_static.columns:
    r = test_returns @ weights_static[name]
    plt.plot(r.index, (1 + r).cumprod(), label=name, alpha=0.85)
plt.title("Out-of-Sample Growth Including CVaR Variants")
plt.xlabel("Date")
plt.ylabel("Growth")
plt.legend(ncol=2)
plt.show()

plt.figure(figsize=(10, 6))
plot_df = test_allocation_summary.sort_values("CVaR")
plt.barh(plot_df["portfolio"], plot_df["CVaR"])
plt.title(f"Out-of-Sample {int(config.alpha*100)}% CVaR")
plt.xlabel("CVaR loss")
plt.ylabel("Portfolio")
plt.show()

## 18. Rolling walk-forward CVaR optimisation

A practical allocator re-estimates CVaR over time.

At each rebalance:

1. use trailing scenario window;
2. solve min-CVaR LP;
3. optionally include turnover penalty;
4. hold until next rebalance;
5. apply weights to next returns.

This is where overfitting and turnover become obvious.

In [ ]:
def rolling_cvar_weights(returns_df, config, method="min_cvar"):
    rows = []
    diag_rows = []

    prev_w = np.ones(len(asset_cols)) / len(asset_cols)
    current = config.rolling_window

    while current < len(returns_df):
        window = returns_df.iloc[current - config.rolling_window:current]

        if method == "min_cvar":
            sol = solve_cvar_lp(
                window,
                alpha=config.alpha,
                min_weight=config.min_weight,
                max_weight=config.max_weight
            )
        elif method == "turnover_aware_cvar":
            sol = solve_cvar_lp_with_turnover(
                window,
                previous_weights=prev_w,
                alpha=config.alpha,
                min_weight=config.min_weight,
                max_weight=config.max_weight,
                turnover_penalty=config.turnover_penalty
            )
        elif method == "stress_augmented_cvar":
            augmented = augment_with_stress_scenarios(
                window,
                stress_scenarios,
                config.stress_scenario_multiplier
            )
            sol = solve_cvar_lp(
                augmented,
                alpha=config.alpha,
                min_weight=config.min_weight,
                max_weight=config.max_weight
            )
        else:
            raise ValueError("Unknown rolling CVaR method.")

        w = pd.Series(sol["weights"], index=asset_cols)
        hold_end = min(current + config.rebalance_step, len(returns_df))

        for date in returns_df.index[current:hold_end]:
            rows.append(pd.Series(w, name=date))

        tail = portfolio_tail_metrics(window, w, config.alpha)

        diag_rows.append({
            "rebalance_date": returns_df.index[current],
            "method": method,
            "solver_success": sol["success"],
            "solver_method": sol["method"],
            "objective": sol["objective"],
            "tau": sol["tau"],
            "train_window_CVaR": tail["CVaR"],
            "train_window_VaR": tail["VaR"],
            "effective_n": float(1.0 / np.sum(w.to_numpy() ** 2)),
            "max_weight": float(w.max()),
            "turnover_vs_previous": float(np.sum(np.abs(w.to_numpy() - prev_w))),
        })

        prev_w = w.to_numpy()
        current += config.rebalance_step

    weights = pd.DataFrame(rows)
    weights.index.name = "date"
    weights = weights.reindex(returns_df.index).ffill().fillna(0.0)

    diagnostics = pd.DataFrame(diag_rows)

    return weights, diagnostics


rolling_methods = ["min_cvar", "turnover_aware_cvar", "stress_augmented_cvar"]

rolling_weights = {}
rolling_diagnostics = {}

for method in rolling_methods:
    w, d = rolling_cvar_weights(returns, config, method=method)
    rolling_weights[method] = w
    rolling_diagnostics[method] = d

rolling_diagnostics["min_cvar"].head()

## 19. Rolling backtest with costs

Weights at $t-1$ are applied to return at $t$.

Transaction cost:

$$
cost_t = c\sum_i |w_{i,t}-w_{i,t-1}|
$$

In [ ]:
def backtest_dynamic_weights(returns_df, weights_df, cost_bps):
    investable = weights_df.shift(1).fillna(0.0)
    gross = (investable * returns_df).sum(axis=1)

    turnover = weights_df.diff().abs().sum(axis=1).fillna(weights_df.abs().sum(axis=1))
    cost = turnover * cost_bps / 10000.0

    net = gross - cost
    nav = (1 + net).cumprod()

    return pd.DataFrame({
        "gross_return": gross,
        "transaction_cost": cost,
        "net_return": net,
        "nav": nav,
        "turnover": turnover,
        "gross_exposure": investable.abs().sum(axis=1),
    })


rolling_backtests = {
    method: backtest_dynamic_weights(returns, weights, config.transaction_cost_bps)
    for method, weights in rolling_weights.items()
}

# Add simple baselines for rolling comparison.
baseline_weights = {
    "equal_weight": pd.DataFrame(np.tile(w_equal.to_numpy(), (len(returns), 1)), index=returns.index, columns=asset_cols),
    "inverse_vol_static": pd.DataFrame(np.tile(w_inverse_vol.to_numpy(), (len(returns), 1)), index=returns.index, columns=asset_cols),
    "simple_hrp_static": pd.DataFrame(np.tile(w_hrp.to_numpy(), (len(returns), 1)), index=returns.index, columns=asset_cols),
}

for method, weights in baseline_weights.items():
    rolling_backtests[method] = backtest_dynamic_weights(returns, weights, config.transaction_cost_bps)


def summarise_backtests(backtests, start_idx):
    rows = []

    for name, bt in backtests.items():
        sub = bt.iloc[start_idx:].copy()
        r = sub["net_return"]
        nav = (1 + r).cumprod()
        dd = nav / nav.cummax() - 1.0
        losses = -r
        var, cvar = historical_var_cvar(losses, config.alpha)

        rows.append({
            "portfolio": name,
            "annualised_return": float(r.mean() * config.annualisation),
            "annualised_vol": float(r.std(ddof=1) * np.sqrt(config.annualisation)),
            "sharpe_proxy": float(r.mean() / r.std(ddof=1) * np.sqrt(config.annualisation)) if r.std(ddof=1) > 0 else np.nan,
            "VaR": var,
            "CVaR": cvar,
            "max_drawdown": float(dd.min()),
            "worst_day": float(r.min()),
            "total_return": float(nav.iloc[-1] - 1.0),
            "mean_turnover": float(sub["turnover"].mean()),
            "annualised_cost_drag": float(sub["transaction_cost"].mean() * config.annualisation),
        })

    return pd.DataFrame(rows).sort_values("CVaR")


rolling_performance = summarise_backtests(rolling_backtests, config.rolling_window)
rolling_performance

In [ ]:
plt.figure(figsize=(12, 6))
for name, bt in rolling_backtests.items():
    sub = bt.iloc[config.rolling_window:]
    nav = (1 + sub["net_return"]).cumprod()
    plt.plot(sub.index, nav, label=name, alpha=0.85)
plt.title("Rolling CVaR Optimisation Backtest")
plt.xlabel("Date")
plt.ylabel("Growth")
plt.legend(ncol=2)
plt.show()

plt.figure(figsize=(10, 6))
plot_df = rolling_performance.sort_values("CVaR")
plt.barh(plot_df["portfolio"], plot_df["CVaR"])
plt.title(f"Rolling Backtest {int(config.alpha*100)}% CVaR")
plt.xlabel("CVaR loss")
plt.ylabel("Portfolio")
plt.show()

## 20. Rolling CVaR weights

Inspect allocation dynamics.

CVaR portfolios often concentrate in assets that had good historical tail behaviour.

This can be sensible or overfit.

In [ ]:
plt.figure(figsize=(12, 6))
for asset in asset_cols:
    plt.plot(rolling_weights["min_cvar"].index, rolling_weights["min_cvar"][asset], label=asset, alpha=0.75)
plt.title("Rolling Min-CVaR Weights")
plt.xlabel("Date")
plt.ylabel("Weight")
plt.legend(ncol=3)
plt.show()

plt.figure(figsize=(12, 5))
for method in rolling_methods:
    plt.plot(rolling_backtests[method].index, rolling_backtests[method]["turnover"].rolling(21).mean(), label=method)
plt.title("Rolling 21-Day Average Turnover")
plt.xlabel("Date")
plt.ylabel("Turnover")
plt.legend()
plt.show()

## 21. Tail exposure attribution

For a portfolio, tail days are the days where loss exceeds VaR.

Asset contribution on a tail day:

$$
contribution_{i,t} = -w_i r_{i,t}
$$

Average tail contribution:

$$
E[-w_i r_{i,t} \mid L_t \ge VaR]
$$

This shows which assets drive CVaR.

In [ ]:
def tail_contribution_table(returns_df, weights, alpha, portfolio_name):
    r = returns_df @ weights
    losses = -r
    var, cvar = historical_var_cvar(losses, alpha)
    tail_mask = losses >= var

    contrib = -returns_df.loc[tail_mask].multiply(weights, axis=1)
    avg_contrib = contrib.mean()

    table = pd.DataFrame({
        "portfolio": portfolio_name,
        "asset": returns_df.columns,
        "weight": weights.reindex(returns_df.columns).to_numpy(),
        "avg_tail_loss_contribution": avg_contrib.reindex(returns_df.columns).to_numpy(),
    })

    table["tail_contribution_pct"] = table["avg_tail_loss_contribution"] / table["avg_tail_loss_contribution"].sum()
    table["VaR"] = var
    table["CVaR"] = cvar
    table["n_tail_days"] = int(tail_mask.sum())

    return table.sort_values("avg_tail_loss_contribution", ascending=False)


tail_contrib_frames = []

for portfolio in ["equal_weight", "simple_hrp", "min_cvar", "stress_augmented_cvar", "turnover_aware_cvar"]:
    tail_contrib_frames.append(
        tail_contribution_table(test_returns, weights_static[portfolio], config.alpha, portfolio)
    )

tail_contributions = pd.concat(tail_contrib_frames, ignore_index=True)
tail_contributions.head()

In [ ]:
portfolio = "min_cvar"
sub = tail_contributions[tail_contributions["portfolio"] == portfolio].sort_values("avg_tail_loss_contribution")

plt.figure(figsize=(10, 6))
plt.barh(sub["asset"], sub["avg_tail_loss_contribution"])
plt.title(f"Average Tail Loss Contribution: {portfolio}")
plt.xlabel("Average contribution on tail days")
plt.ylabel("Asset")
plt.show()

## 22. Stress-regime performance

We compare performance during calm and stress regimes.

CVaR optimisation should reduce stress losses if the training scenarios represent the stress regime.

In [ ]:
regime_series = returns_df.set_index("date")["regime"].reindex(returns.index)
stress_type_series = returns_df.set_index("date")["stress_type"].reindex(returns.index)

stress_rows = []

for name, bt in rolling_backtests.items():
    sub = bt.iloc[config.rolling_window:].copy()
    sub["regime"] = regime_series.reindex(sub.index)
    sub["stress_type"] = stress_type_series.reindex(sub.index)

    for regime_value, group in sub.groupby("regime"):
        stress_rows.append({
            "portfolio": name,
            "regime": int(regime_value),
            "n_days": int(len(group)),
            "mean_return_ann": float(group["net_return"].mean() * config.annualisation),
            "vol_ann": float(group["net_return"].std(ddof=1) * np.sqrt(config.annualisation)),
            "worst_day": float(group["net_return"].min()),
            "mean_turnover": float(group["turnover"].mean()),
        })

stress_regime_report = pd.DataFrame(stress_rows)
stress_regime_report.head()

In [ ]:
plt.figure(figsize=(12, 6))
for name in ["equal_weight", "simple_hrp_static", "min_cvar", "stress_augmented_cvar"]:
    sub = stress_regime_report[stress_regime_report["portfolio"] == name]
    if len(sub):
        plt.plot(sub["regime"], sub["vol_ann"], marker="o", label=name)
plt.title("Realised Volatility by Regime")
plt.xlabel("Regime")
plt.ylabel("Annualised volatility")
plt.xticks([0, 1])
plt.legend()
plt.show()

## 23. Governance flags

CVaR optimisation should be reviewed if:

1. it concentrates too heavily;
2. turnover is excessive;
3. out-of-sample CVaR is not improved;
4. stress losses are large;
5. tail loss is driven by one asset;
6. solver fails frequently.

In [ ]:
equal_cvar = rolling_performance[rolling_performance["portfolio"] == "equal_weight"]["CVaR"].iloc[0]

governance_rows = []

for _, row in rolling_performance.iterrows():
    portfolio = row["portfolio"]

    if portfolio in rolling_weights:
        avg_weights = rolling_weights[portfolio].iloc[config.rolling_window:].mean()
    elif portfolio == "equal_weight":
        avg_weights = w_equal
    elif portfolio == "inverse_vol_static":
        avg_weights = w_inverse_vol
    elif portfolio == "simple_hrp_static":
        avg_weights = w_hrp
    else:
        avg_weights = pd.Series(np.nan, index=asset_cols)

    max_weight = avg_weights.max()
    effective_n = 1.0 / np.sum(avg_weights.dropna().to_numpy() ** 2)

    cvar_improvement = 1 - row["CVaR"] / equal_cvar if equal_cvar != 0 else np.nan

    governance_rows.append({
        "portfolio": portfolio,
        "CVaR": row["CVaR"],
        "cvar_improvement_vs_equal_weight": cvar_improvement,
        "max_avg_weight": max_weight,
        "effective_n_avg_weight": effective_n,
        "mean_turnover": row["mean_turnover"],
        "max_drawdown": row["max_drawdown"],
        "flag_cvar_worse_than_equal_weight": bool(cvar_improvement < 0),
        "flag_max_weight_above_30pct": bool(max_weight > 0.30),
        "flag_effective_n_below_4": bool(effective_n < 4),
        "flag_turnover_above_20pct_daily_avg": bool(row["mean_turnover"] > 0.20),
        "flag_drawdown_below_minus_20pct": bool(row["max_drawdown"] < -0.20),
    })

governance_flags = pd.DataFrame(governance_rows)
governance_flags["review_required"] = governance_flags[
    [
        "flag_cvar_worse_than_equal_weight",
        "flag_max_weight_above_30pct",
        "flag_effective_n_below_4",
        "flag_turnover_above_20pct_daily_avg",
        "flag_drawdown_below_minus_20pct",
    ]
].any(axis=1)

governance_flags

## 24. Audit checks

Numerical checks:

1. weights sum to one;
2. weights respect bounds;
3. CVaR solution has finite objective;
4. LP solver status is reported;
5. rolling backtest returns are finite;
6. tail contributions sum to CVaR approximately.

In [ ]:
audit_rows = []

for portfolio in weights_static.columns:
    w = weights_static[portfolio]

    audit_rows.append({
        "check": f"{portfolio}_weights_sum_to_one",
        "value": float(abs(w.sum() - 1.0)),
        "passed": bool(abs(w.sum() - 1.0) < 1e-8)
    })

    audit_rows.append({
        "check": f"{portfolio}_weights_within_bounds",
        "value": float(max(w.max() - config.max_weight, config.min_weight - w.min())),
        "passed": bool((w.max() <= config.max_weight + 1e-8) and (w.min() >= config.min_weight - 1e-8))
    })

audit_rows.append({
    "check": "min_cvar_solver_status_recorded",
    "value": cvar_solution["method"],
    "passed": bool(cvar_solution["method"] is not None)
})

audit_rows.append({
    "check": "min_cvar_objective_finite",
    "value": float(cvar_solution["objective"]),
    "passed": bool(np.isfinite(cvar_solution["objective"]))
})

finite_backtests = all(np.isfinite(bt["net_return"]).all() for bt in rolling_backtests.values())
audit_rows.append({
    "check": "rolling_backtest_returns_finite",
    "value": bool(finite_backtests),
    "passed": bool(finite_backtests)
})

audit_report = pd.DataFrame(audit_rows)
audit_report

## 25. Practical checklist for CVaR optimisation

Before trusting a CVaR-optimised portfolio:

1. **Check scenario quality**  
   Are historical scenarios representative?

2. **Check stress augmentation**  
   Are rare but plausible shocks included?

3. **Check concentration**  
   Did the optimiser hide in a few historically defensive assets?

4. **Check turnover**  
   Is the tail optimiser unstable through time?

5. **Check out-of-sample CVaR**  
   Did tail risk actually improve after training?

6. **Check expected return trade-off**  
   Is the portfolio too defensive?

7. **Check tail attribution**  
   Which assets dominate CVaR?

8. **Check solver status**  
   Did the LP solve cleanly?

9. **Check constraints**  
   Are weights and exposures realistic?

10. **Remember model risk**  
   CVaR only controls the tail represented by the scenarios.

## 26. Saving outputs

The notebook saves:

1. synthetic returns;
2. factor returns;
3. asset metadata;
4. return summary;
5. static weights;
6. train/test allocation summaries;
7. mean-CVaR frontier;
8. frontier weights;
9. stress scenarios;
10. stress scenario losses;
11. rolling CVaR weights;
12. rolling diagnostics;
13. rolling backtests;
14. rolling performance;
15. tail contribution reports;
16. stress-regime reports;
17. governance flags;
18. audit report;
19. manifest.

In [ ]:
output_dir = Path("data/processed/cvar_convex_optimization_v1")
output_dir.mkdir(parents=True, exist_ok=True)

returns_path = output_dir / "synthetic_returns.csv"
factor_returns_path = output_dir / "factor_returns.csv"
metadata_path = output_dir / "asset_metadata.csv"
return_summary_path = output_dir / "return_summary.csv"
weights_static_path = output_dir / "static_weights.csv"
train_summary_path = output_dir / "train_allocation_summary.csv"
test_summary_path = output_dir / "test_allocation_summary.csv"
frontier_path = output_dir / "mean_cvar_frontier.csv"
frontier_weights_path = output_dir / "mean_cvar_frontier_weights.csv"
stress_scenarios_path = output_dir / "stress_scenarios.csv"
stress_loss_path = output_dir / "stress_loss_table.csv"
rolling_performance_path = output_dir / "rolling_performance.csv"
tail_contributions_path = output_dir / "tail_contributions.csv"
stress_regime_path = output_dir / "stress_regime_report.csv"
governance_flags_path = output_dir / "governance_flags.csv"
audit_report_path = output_dir / "audit_report.csv"
manifest_path = output_dir / "manifest.json"

returns_df.to_csv(returns_path, index=False)
factor_returns.to_csv(factor_returns_path, index=False)
asset_metadata.to_csv(metadata_path, index=False)
return_summary.to_csv(return_summary_path)
weights_static.to_csv(weights_static_path)
train_allocation_summary.to_csv(train_summary_path, index=False)
test_allocation_summary.to_csv(test_summary_path, index=False)
mean_cvar_frontier.to_csv(frontier_path, index=False)
mean_cvar_frontier_weights.to_csv(frontier_weights_path, index=False)
stress_scenarios.to_csv(stress_scenarios_path)
stress_loss_table.to_csv(stress_loss_path, index=False)
rolling_performance.to_csv(rolling_performance_path, index=False)
tail_contributions.to_csv(tail_contributions_path, index=False)
stress_regime_report.to_csv(stress_regime_path, index=False)
governance_flags.to_csv(governance_flags_path, index=False)
audit_report.to_csv(audit_report_path, index=False)

rolling_weight_paths = {}
rolling_diag_paths = {}
rolling_bt_paths = {}

for method, weights in rolling_weights.items():
    path = output_dir / f"rolling_weights_{method}.csv"
    weights.to_csv(path)
    rolling_weight_paths[method] = str(path)

for method, diag in rolling_diagnostics.items():
    path = output_dir / f"rolling_diagnostics_{method}.csv"
    diag.to_csv(path, index=False)
    rolling_diag_paths[method] = str(path)

for method, bt in rolling_backtests.items():
    path = output_dir / f"rolling_backtest_{method}.csv"
    bt.to_csv(path)
    rolling_bt_paths[method] = str(path)

manifest = {
    "dataset_name": "cvar_convex_optimization_outputs",
    "schema_version": "cvar_convex_optimization_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "asset_cols": asset_cols,
    "split_metadata": split_metadata,
    "scipy_available": SCIPY_AVAILABLE,
    "sklearn_available": SKLEARN_AVAILABLE,
    "cvar_solution": {
        "success": cvar_solution["success"],
        "method": cvar_solution["method"],
        "objective": cvar_solution["objective"],
        "message": cvar_solution["message"],
    },
    "stress_cvar_solution": {
        "success": stress_cvar_solution["success"],
        "method": stress_cvar_solution["method"],
        "objective": stress_cvar_solution["objective"],
        "message": stress_cvar_solution["message"],
    },
    "turnover_cvar_solution": {
        "success": turnover_cvar_solution["success"],
        "method": turnover_cvar_solution["method"],
        "objective": turnover_cvar_solution["objective"],
        "message": turnover_cvar_solution["message"],
    },
    "rolling_performance": rolling_performance.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "rolling_weight_files": rolling_weight_paths,
    "rolling_diagnostic_files": rolling_diag_paths,
    "rolling_backtest_files": rolling_bt_paths,
    "notes": [
        "CVaR optimisation uses the Rockafellar-Uryasev linear programming formulation.",
        "Variables are portfolio weights, VaR threshold tau, and tail slack variables u_s.",
        "Target-return CVaR optimisation is used to build a mean-CVaR frontier.",
        "Stress-scenario augmentation repeats explicit stress rows to increase their importance in the scenario set.",
        "Turnover-aware CVaR optimisation adds auxiliary variables for absolute weight changes.",
        "Rolling CVaR optimisation includes transaction costs and weights are shifted by one day before applying returns.",
        "CVaR controls only the tail represented by the scenario set; unseen regimes remain model risk."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

returns_path, weights_static_path, rolling_performance_path, governance_flags_path, manifest_path

## 27. Limitations

### 27.1 Synthetic data

The notebook uses synthetic returns and synthetic stress scenarios.

Real portfolios require clean historical data, realistic stress libraries, holdings, derivatives, liquidity, and transaction costs.

### 27.2 Scenario dependence

CVaR optimisation only controls losses in the scenario set.

If future tail events are different, the optimiser may fail.

### 27.3 Historical overfitting

A CVaR optimiser can overfit rare historical tail events.

Stress augmentation helps but introduces subjective scenario choices.

### 27.4 Expected returns remain noisy

Target-return constraints use sample mean estimates, which are unreliable.

### 27.5 Long-only constraints

The notebook uses long-only weights.

Long-short CVaR optimisation requires leverage and margin controls.

### 27.6 Simplified transaction costs

Transaction costs are fixed basis points.

Real costs depend on liquidity, volatility, market impact, and execution.

### 27.7 No derivative convexity

The notebook optimises asset returns directly.

Options and nonlinear derivatives require scenario pricing, not linear returns.

### 27.8 CVaR is not maximum loss

CVaR is average tail loss beyond VaR, not a worst-case bound.

## 28. Research frontier and extensions

Important extensions include:

1. **Robust CVaR optimisation**  
   Use uncertainty sets around scenarios.

2. **Distributionally robust optimisation**  
   Optimise worst-case CVaR over probability distributions.

3. **CVaR with factor scenarios**  
   Generate scenarios from factor models.

4. **CVaR with option portfolios**  
   Price nonlinear instruments under each scenario.

5. **Liquidity-adjusted CVaR**  
   Include liquidation costs and market impact.

6. **Drawdown-constrained optimisation**  
   Optimise path-dependent risk.

7. **Multi-period CVaR**  
   Optimise over scenario trees.

8. **CVaR risk budgeting**  
   Equalise tail-risk contributions.

9. **Regime-conditioned CVaR**  
   Optimise differently in calm versus stress regimes.

10. **Chinese futures CVaR allocation**  
   Include price limits, night sessions, margin, contract rolls, and commodity-specific stress scenarios.

## 29. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_12_black_litterman_portfolio_construction**  
   Stabilise expected returns using priors and views.

2. **04_13_correlation_breakdown_and_regime_risk**  
   Tail risk under correlation shifts.

3. **04_14_portfolio_constraints_margin_and_leverage**  
   Add real-world constraints.

4. **05_01_vectorized_backtest_engine**  
   Turn CVaR allocation into a reusable module.

5. **05_06_walk_forward_model_validation**  
   Validate rolling CVaR allocation.

6. **07_02_tail_risk_overlay_capstone**  
   Combine CVaR allocation with option hedging.

## 30. Summary

This notebook implemented CVaR convex optimisation.

It showed how to:

1. define VaR and CVaR;
2. formulate CVaR minimisation as a linear programme;
3. solve long-only minimum-CVaR allocation;
4. compare against equal weight, inverse vol, GMV, and HRP-style baselines;
5. test static portfolios out of sample;
6. build a mean-CVaR efficient frontier;
7. augment the scenario set with stress scenarios;
8. implement turnover-aware CVaR optimisation;
9. run rolling CVaR allocation with transaction costs;
10. inspect allocation dynamics;
11. attribute tail loss contributions;
12. evaluate stress-regime performance;
13. create governance flags and audit checks;
14. save outputs and metadata.

The key computational takeaway:

> Empirical CVaR portfolio optimisation can be written as a linear programme with auxiliary tail-loss variables.

The key financial takeaway:

> CVaR optimisation targets downside severity directly, but it is only as good as the scenarios used to define the tail.

## 31. Further reading

- Rockafellar and Uryasev, "Optimization of Conditional Value-at-Risk."
- Acerbi and Tasche on Expected Shortfall.
- McNeil, Frey, and Embrechts, *Quantitative Risk Management.*
- Meucci, *Risk and Asset Allocation.*
- Literature on convex risk measures, distributionally robust optimisation, and scenario-based portfolio optimisation.